# Proyecto Final - Detección de Elementos de Seguridad en Obras

Autores:
- Jesús Matheus
- Octavio Marchant
- Martin Jofre

Objetivo:
Comparar tres enfoques para detectar trabajadores sin casco en un entorno de construcción:

1. YOLOv8 (modelo entrenado)
2. YOLO-World (vocabulario abierto)
3. Florence-2 (Vision Language Model)

Todas las pruebas fueron realizadas en Google Colab utilizando GPU NVIDIA T4.

##### Prueba de tarjeta

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available()
else 'sin GPU')

True Tesla T4


Consumo de API Roboflow

In [ ]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 104.1 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92


In [ ]:
from roboflow import Roboflow
from google.colab import userdata
rf = Roboflow(api_key=userdata.get('roboflow'))
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
dataset = project.version(27).download("yolov8") # ajusta el número de versión al del snippet
print(dataset.location) # carpeta con train/ valid/ test/ y data.yaml


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Construction-Site-Safety-27 in yolov8:: 100%|██████████| 5610/5610 [00:02<00:00, 2120.09it/s]


/content/Construction-Site-Safety-27


Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('hf'))

Fijación de semilla

In [ ]:
import torch
import numpy as np
import random

# Fijar semillas (SEEDS) para asegurar reproducibilidad (REPRODUCIBILITY)
# Esto garantiza que las divisiones internas y el comportamiento de la red neuronal (NEURAL NETWORK) sean deterministas (DETERMINISTIC)
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"✅ Semillas (SEEDS) fijadas globalmente en el valor: {SEED}")

✅ Semillas (SEEDS) fijadas globalmente en el valor: 42


# Yolo V8

## Cargamos modelo base

In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.7 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Comprobación de dataset

In [ ]:
import os

os.path.exists("/content/Construction-Site-Safety-27/data.yaml")

True

In [ ]:
with open("/content/Construction-Site-Safety-27/data.yaml") as f:
    print(f.read())

names:
- Hardhat
- Mask
- NO-Hardhat
- NO-Mask
- NO-Safety Vest
- Person
- Safety Cone
- Safety Vest
- machinery
- vehicle
nc: 10
roboflow:
  license: CC BY 4.0
  project: construction-site-safety
  url: https://universe.roboflow.com/roboflow-universe-projects/construction-site-safety/dataset/27
  version: 27
  workspace: roboflow-universe-projects
test: ../test/images
train: ../train/images
val: ../valid/images



## Entrenamiento del modelo

In [ ]:
model = YOLO("yolov8n.pt")

In [ ]:
results = model.train(
    data="/content/Construction-Site-Safety-27/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0
)

Ultralytics 8.4.91 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Construction-Site-Safety-27/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=aut

## Guardar modelo en drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p "/content/drive/MyDrive/Proyecto_Final_OFC"

In [ ]:
!cp -r /content/runs "/content/drive/MyDrive/Proyecto_Final_OFC/"

In [ ]:
!ls "/content/drive/MyDrive/Proyecto_Final_OFC/runs/detect"

train


##  Cargar modelo entrenado

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/Proyecto_Final_OFC/runs/detect/train/weights/best.pt")

Mounted at /content/drive


## Evaluamos sobre test

In [ ]:
metrics = model.val(
    data="/content/Construction-Site-Safety-27/data.yaml",
    split="test"
)

Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1262.4±462.6 MB/s, size: 53.6 KB)
val: Scanning /content/Construction-Site-Safety-27/test/labels... 82 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 82/82 1.2Kit/s 0.1s
val: New cache created: /content/Construction-Site-Safety-27/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.0it/s 5.8s
                   all         82        760      0.827      0.651      0.701      0.393
               Hardhat         30        110      0.957      0.755      0.858      0.514
                  Mask         16         28      0.979      0.714      0.728      0.475
            NO-Hardhat         25         41      0.771      0.561      0.541       0.27
               NO-Mask         30         79      0.

In [ ]:
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precisión:", metrics.box.mp)
print("Recall:", metrics.box.mr)

mAP50: 0.7005799752177349
mAP50-95: 0.3934793829540024
Precisión: 0.8269796251723935
Recall: 0.650982445375295


## Resultados por clase

In [ ]:
print(f"{'Clase':20} {'AP50-95':>10}")

for i, ap in enumerate(metrics.box.maps):
    print(f"{model.names[i]:20} {ap:.4f}")

Clase                   AP50-95
Hardhat              0.5137
Mask                 0.4752
NO-Hardhat           0.2702
NO-Mask              0.2777
NO-Safety Vest       0.3828
Person               0.4526
Safety Cone          0.1415
Safety Vest          0.5082
machinery            0.5184
vehicle              0.3944


In [ ]:
print("--- AP@0.5 CLASES COMPARTIBLES (SHARED CLASSES) ---")
print(f"{'Clase (CLASS)':20} {'AP@0.5':>10}")

# Iteramos sobre el número de clases (nc) que tiene el modelo
for i in range(metrics.box.nc):
    # Obtenemos el nombre de la clase
    class_name = metrics.names[i]

    # Filtramos solo las clases que nos interesan
    if class_name in ["Person", "Hardhat"]:
        # Usamos el método class_result(i) que devuelve: p, r, ap50, ap
        # Por lo tanto, el índice 2 es ap50
        _, _, ap50, _ = metrics.box.class_result(i)
        print(f"{class_name:20} {ap50:.4f}")

--- AP@0.5 CLASES COMPARTIBLES (SHARED CLASSES) ---
Clase (CLASS)            AP@0.5
Hardhat              0.8580
Person               0.7734


## f1 a nivel

In [ ]:
from pathlib import Path
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# -------------------------
# Configuración
# -------------------------
TEST_IMAGES = Path("/content/Construction-Site-Safety-27/test/images")
TEST_LABELS = Path("/content/Construction-Site-Safety-27/test/labels")

NO_HARDHAT_CLASS = 2
CONF_THRESHOLD = 0.25

y_true = []
y_pred = []

# -------------------------
# Recorrer imágenes
# -------------------------
for img_path in sorted(TEST_IMAGES.glob("*")):

    label_path = TEST_LABELS / (img_path.stem + ".txt")

    # ---------- Ground Truth ----------
    gt_no_hardhat = False

    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                class_id = int(line.split()[0])
                if class_id == NO_HARDHAT_CLASS:
                    gt_no_hardhat = True
                    break

    # ---------- Predicción ----------
    results = model.predict(
        source=str(img_path),
        conf=CONF_THRESHOLD,
        verbose=False
    )

    pred_no_hardhat = False

    for cls in results[0].boxes.cls.tolist():
        if int(cls) == NO_HARDHAT_CLASS:
            pred_no_hardhat = True
            break

    y_true.append(gt_no_hardhat)
    y_pred.append(pred_no_hardhat)

# -------------------------
# Métricas
# -------------------------

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("="*40)
print("F1 A NIVEL DE EVENTO")
print("="*40)

print(f"TP : {tp}")
print(f"FP : {fp}")
print(f"TN : {tn}")
print(f"FN : {fn}")

print()

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

F1 A NIVEL DE EVENTO
TP : 19
FP : 1
TN : 56
FN : 6

Precision : 0.9500
Recall    : 0.7600
F1 Score  : 0.8444


Aunque el desempeño de YOLOv8 sobre la clase NO-Hardhat fue moderado a nivel de detección de objetos (AP@0.5:0.95 = 0.2702), el sistema logró identificar correctamente la mayoría de los eventos de interés ("existe al menos una persona sin casco") alcanzando un F1 de 0.8444.

## Velocidad

warm-up

In [ ]:
# Warm-up (no se mide)
first_image = sorted(TEST_IMAGES.glob("*"))[0]

model.predict(
    source=str(first_image),
    conf=0.25,
    verbose=False
)

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Hardhat', 1: 'Mask', 2: 'NO-Hardhat', 3: 'NO-Mask', 4: 'NO-Safety Vest', 5: 'Person', 6: 'Safety Cone', 7: 'Safety Vest', 8: 'machinery', 9: 'vehicle'}
 obb: None
 orig_img: array([[[ 81, 113, 124],
         [ 75, 105, 116],
         [ 66,  94, 105],
         ...,
         [ 64,  60,  72],
         [ 58,  54,  66],
         [ 54,  50,  62]],
 
        [[ 77, 109, 120],
         [ 73, 103, 114],
         [ 67,  95, 106],
         ...,
         [ 58,  54,  66],
         [ 54,  50,  62],
         [ 52,  48,  60]],
 
        [[ 81, 113, 124],
         [ 81, 111, 122],
         [ 79, 108, 117],
         ...,
         [ 44,  42,  54],
         [ 45,  41,  53],
         [ 44,  40,  52]],
 
        ...,
 
        [[ 20,  20,  20],
         [ 22,  22,  22],
         [ 23,  23,  23],
         ...,
         [ 62,  66,  67],
         [ 61,  66,  6

Medicion

In [ ]:
import time
from pathlib import Path
import numpy as np

TEST_IMAGES = Path("/content/Construction-Site-Safety-27/test/images")

times = []

for img_path in sorted(TEST_IMAGES.glob("*")):

    start = time.perf_counter()

    model.predict(
        source=str(img_path),
        conf=0.25,
        verbose=False
    )

    end = time.perf_counter()

    times.append(end - start)

latency = np.mean(times)
fps = 1 / latency

print("="*40)
print("VELOCIDAD DEL MODELO")
print("="*40)
print(f"Frames evaluados : {len(times)}")
print(f"Latencia promedio: {latency:.4f} s/frame")
print(f"FPS promedio     : {fps:.2f}")

VELOCIDAD DEL MODELO
Frames evaluados : 82
Latencia promedio: 0.0140 s/frame
FPS promedio     : 71.32


## Robustez

In [ ]:
import requests
from PIL import Image
import io

print("--- PRUEBA DE ROBUSTEZ CUANTITATIVA (ROBUSTNESS TEST) - YOLOv8 ---")
print("Escenario planteado: La empresa solicita mañana detectar 'guantes' (GLOVES).")
print("Evaluando el modelo cerrado sobre imágenes fuera de dominio con la nueva clase...")

# URLs de 3 imágenes de stock reales con trabajadores usando guantes de seguridad
urls_guantes = [
    "https://images.pexels.com/photos/8961403/pexels-photo-8961403.jpeg",  # Primer plano de manos con guantes en obra
    "https://images.pexels.com/photos/6195125/pexels-photo-6195125.jpeg",  # Trabajador con herramientas y guantes puestos
    "https://images.pexels.com/photos/8961131/pexels-photo-8961131.jpeg"   # Operario manejando material con guantes
]

for idx, url in enumerate(urls_guantes, 1):
    try:
        response = requests.get(url, timeout=10)
        img = Image.open(io.BytesIO(response.content)).convert("RGB")

        # Inferencia con el modelo cerrado entrenado en CSS-Data
        res = model.predict(source=img, conf=0.25, verbose=False)
        clases_detectadas = [model.names[int(c)] for c in res[0].boxes.cls.tolist()]

        print(f"\n📷 Imagen de prueba {idx} (Guantes de seguridad):")
        print(f"   -> Clases forzadas/alucinadas: {clases_detectadas}")
    except Exception as e:
        print(f"Error al descargar la imagen {idx}: {e}")

# Nota para el informe final:
# Al ser un detector cerrado, YOLOv8 posee un vocabulario rígido. Ante la presencia
# de 'guantes', el modelo es incapaz de reconocerlos o, peor aún, puede alucinar forzando
# las etiquetas que sí conoce (como 'Mask' o 'Safety Vest') debido a similitudes de textura[cite: 2].

--- PRUEBA DE ROBUSTEZ CUANTITATIVA (ROBUSTNESS TEST) - YOLOv8 ---
Escenario planteado: La empresa solicita mañana detectar 'guantes' (GLOVES).
Evaluando el modelo cerrado sobre imágenes fuera de dominio con la nueva clase...

📷 Imagen de prueba 1 (Guantes de seguridad):
   -> Clases forzadas/alucinadas: ['person']

📷 Imagen de prueba 2 (Guantes de seguridad):
   -> Clases forzadas/alucinadas: ['person', 'person', 'person']

📷 Imagen de prueba 3 (Guantes de seguridad):
   -> Clases forzadas/alucinadas: ['person', 'person', 'hard hat', 'hard hat']


###  Análisis Cuantitativo de Robustez ante Nuevos Requerimientos (Clase Imprevista)

El experimento cuantitativo con imágenes de guantes de seguridad expone de manera empírica la limitación fundamental del enfoque de detector cerrado (CLOSED DETECTOR):

#### 1. Ceguera Semántica Absoluta
Al no haber sido entrenado con la clase explícita `Gloves`, el modelo es matemáticamente incapaz de generar una caja delimitadora (BOUNDING BOX) correcta para este elemento. En las imágenes 1 y 2, el sistema ignoró por completo las manos del operario y solo localizó la entidad macroconocida `Person`

#### 2. Riesgo de Alucinación en Producción (El Caso Crítico de la Imagen 3)
Ante características visuales no estructuradas o fuera de dominio, el modelo cerrado tiende a forzar sus clases conocidas sobre texturas ambiguas. En la imagen de prueba 3, el detector alucinó e identificó falsamente dos instancias de la clase `Hardhat` sobre los guantes de protección.
*   **Impacto operativo:** En un entorno de producción real, esto se traduce en un **falso negativo encubierto**: el sistema reportará falsamente que los operarios llevan cascos de seguridad puestos cuando en realidad solo están manipulando herramientas con guantes, comprometiendo la integridad física en la obra.

#### 3. Impacto en el Ciclo de Vida del Proyecto
Si la constructora solicita mañana expandir el sistema para vigilar el uso de guantes, el despliegue con YOLOv8 exige detener la escalabilidad del sistema para:
*   Recolectar un nuevo lote de datos masivo en terreno.
*   Etiquetar manualmente miles de instancias de la nueva clase.
*   Modificar la arquitectura en el archivo `data.yaml` y volver a gastar horas de cómputo en realizar un proceso de fine-tuning completo desde cero.

Esto demuestra que, si bien el detector cerrado posee una velocidad de inferencia inigualable (71.32 FPS), carece por completo de la flexibilidad adaptativa que requiere un entorno industrial dinámico.

# Yolo World

## Instalación

In [ ]:
!pip install -q ultralytics supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 5.3 MB/s eta 0:00:00


## Cargamos modelo

In [ ]:
from ultralytics import YOLOWorld


model_world = YOLOWorld("yolov8s-worldv2.pt")
print("Modelo YOLO-World cargado correctamente.")

Modelo YOLO-World cargado correctamente.


## Definición de clases

In [ ]:
classes_world = ["hard hat", "head", "person"]
model_world.set_classes(classes_world)
print("Clases de texto configuradas:", classes_world)

Clases de texto configuradas: ['hard hat', 'head', 'person']


## Inferencia y normalización de clases

In [ ]:
from pathlib import Path

TEST_IMAGES = Path("/content/Construction-Site-Safety-27/test/images")
TEST_LABELS = Path("/content/Construction-Site-Safety-27/test/labels")

NO_HARDHAT_CLASS_GT = 2  # ID según tu archivo data.yaml

y_true_world = []
y_pred_world = []

for img_path in sorted(TEST_IMAGES.glob("*")):
    label_path = TEST_LABELS / (img_path.stem + ".txt")

    # --- Extracción de etiquetas verdaderas (GROUND TRUTH) ---
    gt_no_hardhat = False
    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                if int(line.split()[0]) == NO_HARDHAT_CLASS_GT:
                    gt_no_hardhat = True
                    break

    # --- Inferencia (INFERENCE) con YOLO-World ---
    results = model_world.predict(source=str(img_path), conf=0.25, verbose=False)
    pred_classes = [model_world.names[int(cls)] for cls in results[0].boxes.cls.tolist()]

    # --- Normalización de clases (Lógica de negación) ---
    # Marcamos el cuadro (FRAME) como positivo para la infracción si detecta "head"
    # o si detecta "person" pero no "hard hat".
    pred_no_hardhat = False
    if "head" in pred_classes or ("person" in pred_classes and "hard hat" not in pred_classes):
        pred_no_hardhat = True

    y_true_world.append(gt_no_hardhat)
    y_pred_world.append(pred_no_hardhat)

## métricas

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

tn_w, fp_w, fn_w, tp_w = confusion_matrix(y_true_world, y_pred_world).ravel()
precision_w = precision_score(y_true_world, y_pred_world, zero_division=0)
recall_w = recall_score(y_true_world, y_pred_world, zero_division=0)
f1_w = f1_score(y_true_world, y_pred_world, zero_division=0)

print("--- MÉTRICAS (METRICS) A NIVEL-EVENTO (YOLO-WORLD) ---")
print(f"TP : {tp_w} | FP : {fp_w} | TN : {tn_w} | FN : {fn_w}")
print(f"Precisión (PRECISION): {precision_w:.4f}")
print(f"Sensibilidad (RECALL): {recall_w:.4f}")
print(f"Puntuación F1 (F1 SCORE): {f1_w:.4f}")

--- MÉTRICAS (METRICS) A NIVEL-EVENTO (YOLO-WORLD) ---
TP : 22 | FP : 24 | TN : 33 | FN : 3
Precisión (PRECISION): 0.4783
Sensibilidad (RECALL): 0.8800
Puntuación F1 (F1 SCORE): 0.6197


A diferencia de YOLOv8, YOLO-World logra una sensibilidad (RECALL) muy alta de 0.8800, pero su precisión (PRECISION) decae drásticamente a 0.4783. Esto ocurre porque el paradigma de vocabulario abierto (OPEN VOCABULARY) genera numerosos falsos positivos (FALSE POSITIVES) al inferir indirectamente la regla de seguridad ("head" presente o "person" sin "hard hat"). Queda en evidencia que, aunque es altamente flexible sin necesidad de reentrenamiento, este sistema tiene dificultades inherentes para razonar sobre atributos lógicos directos y escenarios de negación (NEGATION) en la misma escena.

In [ ]:
print("--- EVALUACIÓN DE CAJAS (M mAP@0.5) EN YOLO-WORLD ---")

# 1. Reconfiguramos las clases en el orden EXACTO de tu data.yaml original
# Esto es vital para que las predicciones cuadren con los índices del Ground Truth
clases_originales = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle"
]
model_world.to('cpu')
model_world.set_classes(clases_originales)
model_world.to('cuda')

# 2. Ejecutamos la validación oficial sobre el split de test
metrics_world = model_world.val(
    data="/content/Construction-Site-Safety-27/data.yaml",
    split="test",
    device=0,
    verbose=False
)

print("\n--- AP@0.5 CLASES COMPARTIBLES EN YOLO-WORLD ---")
print(f"{'Clase (CLASS)':20} {'AP@0.5':>10}")

# 3. Extraemos los resultados usando el método simétrico class_result()
for i in range(metrics_world.box.nc):
    class_name = metrics_world.names[i]
    if class_name in ["Person", "Hardhat"]:
        _, _, ap50, _ = metrics_world.box.class_result(i)
        print(f"{class_name:20} {ap50:.4f}")

# 4. Dejamos el modelo listo con sus clases de análisis de negación por si acaso
model_world.to('cpu')
model_world.set_classes(["hard hat", "head", "person"])
model_world.to('cuda');

--- EVALUACIÓN DE CAJAS (M mAP@0.5) EN YOLO-WORLD ---
Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s-worldv2 summary: 236 layers, 164,026,601 parameters, 0 gradients, 34.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1353.6±287.9 MB/s, size: 51.2 KB)
val: Scanning /content/Construction-Site-Safety-27/test/labels.cache... 82 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 82/82 13.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.2it/s 4.8s
                   all         82        760      0.487      0.257      0.321      0.212
Speed: 6.7ms preprocess, 14.4ms inference, 0.0ms loss, 4.5ms postprocess per image
Results saved to /content/runs/detect/val-2

--- AP@0.5 CLASES COMPARTIBLES EN YOLO-WORLD ---
Clase (CLASS)            AP@0.5
Hardhat              0.6480
Person               0.7725


## Velocidad

In [ ]:
# Calentamiento (WARM-UP)
# Ejecutar la primera imagen estabiliza la gráfica (GPU) y evita latencias irreales en el bucle
first_image_world = sorted(TEST_IMAGES.glob("*"))[0]
model_world.predict(source=str(first_image_world), conf=0.25, verbose=False)
print("Calentamiento (WARM-UP) completado.")

Calentamiento (WARM-UP) completado.


In [ ]:
import time
import numpy as np

times_world = []

for img_path in sorted(TEST_IMAGES.glob("*")):
    start = time.perf_counter()
    model_world.predict(source=str(img_path), conf=0.25, verbose=False)
    end = time.perf_counter()
    times_world.append(end - start)

latency_world = np.mean(times_world)
fps_world = 1 / latency_world

print("="*40)
print("VELOCIDAD DEL MODELO YOLO-WORLD")
print("="*40)
print(f"Frames evaluados : {len(times_world)}")
print(f"Latencia promedio: {latency_world:.4f} s/frame")
print(f"FPS promedio     : {fps_world:.2f}")

VELOCIDAD DEL MODELO YOLO-WORLD
Frames evaluados : 82
Latencia promedio: 0.0272 s/frame
FPS promedio     : 36.75


## Robustez

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

print("--- PRUEBA DE ROBUSTEZ CUANTITATIVA (ROBUSTNESS TEST) ---")
print("Evaluando sensibilidad del F1 score frente a variaciones de sintaxis en el indicador de texto (TEXT PROMPT)...")

prompts_alternativos = [
    ["safety helmet", "bare head", "human"],
    ["cap", "person"],
    ["protective headwear", "unprotected head"]
]

resultados_sensibilidad = {}

# Iteramos sobre cada variante de indicador para procesar secuencialmente todo el split de prueba
for prompt in prompts_alternativos:
    # Solución al desajuste: Movemos a CPU para inyectar textos y luego a GPU
    model_world.to('cpu')
    model_world.set_classes(prompt)
    model_world.to('cuda')

    y_pred_variant = []

    # Recorremos exactamente las mismas imágenes del conjunto de test para evaluar varianza real
    for img_path in sorted(TEST_IMAGES.glob("*")):
        res = model_world.predict(source=str(img_path), conf=0.25, verbose=False, device=0)

        # Mapeamos las clases dinámicas detectadas según el prompt actual
        pred_classes = [model_world.names[int(c)] for c in res[0].boxes.cls.tolist()]

        # Aplicamos de forma simétrica la lógica de normalización / negación
        # Buscamos el segundo elemento (la cabeza descubierta) o el humano sin el casco (primer elemento)
        casco_syn = prompt[0]
        cabeza_syn = prompt[1]
        persona_syn = prompt[2] if len(prompt) > 2 else "person"

        pred_no_hardhat = False
        if cabeza_syn in pred_classes or (persona_syn in pred_classes and casco_syn not in pred_classes):
            pred_no_hardhat = True

        y_pred_variant.append(pred_no_hardhat)

    # Calculamos el F1 score de esta variante usando las etiquetas reales acumuladas en y_true_world
    f1_variante = f1_score(y_true_world, y_pred_variant, zero_division=0)
    resultados_sensibilidad[str(prompt)] = round(f1_variante, 4)
    print(f"✅ Variante evaluada {prompt} -> F1 Score: {f1_variante:.4f}")

# Dejamos el modelo limpio restaurando las clases base del proyecto
model_world.to('cpu')
model_world.set_classes(classes_world)
model_world.to('cuda')

print("\n--- RESUMEN DE SENSIBILIDAD AL PROMPT ---")
print(resultados_sensibilidad)

--- PRUEBA DE ROBUSTEZ CUANTITATIVA (ROBUSTNESS TEST) ---
Evaluando sensibilidad del F1 score frente a variaciones de sintaxis en el indicador de texto (TEXT PROMPT)...
✅ Variante evaluada ['safety helmet', 'bare head', 'human'] -> F1 Score: 0.2000
✅ Variante evaluada ['cap', 'person'] -> F1 Score: 0.5897
✅ Variante evaluada ['protective headwear', 'unprotected head'] -> F1 Score: 0.0000

--- RESUMEN DE SENSIBILIDAD AL PROMPT ---
{"['safety helmet', 'bare head', 'human']": 0.2, "['cap', 'person']": 0.5897, "['protective headwear', 'unprotected head']": 0.0}


Análisis Cuantitativo de Sensibilidad Semántica y Robustez
Los resultados de la prueba de robustez cuantitativa demuestran una vulnerabilidad crítica en el paradigma de vocabulario abierto (OPEN VOCABULARY):

1. **Varianza Extrema del Prompt (Prompt Variance):** La variación de sinónimos semánticos equivalentes destruyó la consistencia del sistema. La variante `['protective headwear', 'unprotected head']` arrojó un **0.0000 de F1 Score**, lo que implica una ceguera operativa absoluta ante las infracciones de seguridad en la obra.
2. **Dependencia de Conceptos Genéricos:** La variante `['cap', 'person']` retuvo un desempeño decente (**0.5897**) debido a que "cap" (gorra/capucha) y "person" son tokens masivamente representados en el preentrenamiento de CLIP en YOLO-World, pero sigue estando por debajo del indicador optimizado base[cite: 3].
3. **Impacto en Producción:** Estos datos demuestran que un sistema de vocabulario abierto no puede ser desplegado a ciegas en entornos de alta responsabilidad[cite: 1]. Una sutil actualización en la sintaxis de las clases por parte de un operador en producción podría desactivar por completo la capacidad del modelo para emitir alertas de seguridad, introduciendo falsos negativos críticos para la vida de los trabajadores[cite: 1].

#  Florence-2

## Instalación

In [ ]:
# IMPORTANTE: Florence-2 usa código remoto (trust_remote_code=True) que no se ha
# actualizado desde 2024 y se rompe con transformers >= 4.52 (error típico:
# "Florence2LanguageConfig object has no attribute 'forced_bos_token_id'").
# Fijamos una versión compatible conocida (issue documentado en HF/transformers #36886).
# huggingface_hub 0.34.0 satisface lo que pide transformers==4.49.0 (<1.0) y
# también lo que pide diffusers (>=0.34.0), preinstalado en Colab, para evitar
# el warning de conflicto de dependencias (gradio pide >=1.2, pero no lo usamos
# en este notebook, así que ese conflicto puntual es inofensivo e ignorable).
!pip install -q "transformers==4.49.0" "huggingface_hub==0.34.0" accelerate einops timm
# Nota: NO instalamos flash_attn (falla el build en Colab). Florence-2 lo intenta
# importar igual dentro de su código remoto; lo evitamos con un parche en la
# celda siguiente para que use la atención "eager" en su lugar.
#
# ⚠️ Después de correr esta celda: Entorno de ejecución > Reiniciar sesión,
# y luego correr de nuevo desde aquí hacia abajo. Colab ya tenía cargada una
# versión distinta de transformers en memoria y el pip install solo no basta.

## Cargamos modelo

In [ ]:
import torch
from unittest.mock import patch
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports

# Florence-2 (Microsoft, licencia MIT) es un VLM de grounding pequeño (~230M params
# en su versión "base"), pensado para correr en una T4 sin problema (a diferencia
# de VLMs generativos más grandes como Qwen2.5-VL).
FLORENCE_MODEL_ID = "microsoft/Florence-2-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

# Parche conocido: el código remoto de Florence-2 intenta importar flash_attn
# aunque no esté instalado. Lo removemos de la lista de imports detectados.
def fixed_get_imports(filename):
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    model_florence = AutoModelForCausalLM.from_pretrained(
        FLORENCE_MODEL_ID,
        torch_dtype=torch_dtype,
        trust_remote_code=True,
        attn_implementation="eager",
    ).to(device).eval()

processor_florence = AutoProcessor.from_pretrained(FLORENCE_MODEL_ID, trust_remote_code=True)

print(f"Modelo Florence-2 ({FLORENCE_MODEL_ID}) cargado correctamente en {device}. Licencia: MIT.")


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


Modelo Florence-2 (microsoft/Florence-2-base) cargado correctamente en cuda. Licencia: MIT.


## Definición de tarea y vocabulario (prompt de detección)

In [ ]:
# Tarea de detección de vocabulario abierto (OPEN VOCABULARY DETECTION):
# le damos como texto las mismas clases usadas en YOLO-World, para que la
# comparación entre paradigmas sea justa (mismo vocabulario, mismos frames).
TASK_PROMPT_FLORENCE = "<OPEN_VOCABULARY_DETECTION>"
classes_florence = ["hard hat", "bare head", "person"]

def run_florence_detection(image, text_input, task_prompt=TASK_PROMPT_FLORENCE):
    """Corre una imagen por Florence-2 y devuelve boxes (xyxy) + labels de texto."""
    prompt = task_prompt + text_input
    inputs = processor_florence(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)

    generated_ids = model_florence.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3,
        do_sample=False,
    )
    generated_text = processor_florence.batch_decode(generated_ids, skip_special_tokens=False)[0]

    parsed = processor_florence.post_process_generation(
        generated_text, task=task_prompt, image_size=(image.width, image.height)
    )
    result = parsed[task_prompt]

    boxes = result.get("bboxes", [])
    labels = result.get("bboxes_labels", result.get("labels", []))
    return boxes, labels

print("Prompt de detección configurado:", text_input_florence)

Prompt de detección configurado: hard hat. head. person.


## Inferencia sobre el conjunto de test

Florence-2 es mucho más lento que YOLOv8/YOLO-World (la guía advierte: *"baja (<1-2 FPS)"*),
por lo que corremos **un único loop** sobre `test/images` que guarda las cajas detectadas y
mide el tiempo por frame al mismo tiempo. Así evitamos correr la inferencia dos veces
(una para métricas, otra para velocidad) sobre un modelo caro de ejecutar.

warm-up

In [ ]:
from pathlib import Path
from PIL import Image

def run_florence_multi(image, clases, task_prompt=TASK_PROMPT_FLORENCE):
    all_boxes, all_labels = [], []
    for clase in clases:
        boxes, labels = run_florence_detection(image, clase)  # una frase a la vez
        all_boxes.extend(boxes)
        all_labels.extend([clase.lower().strip()] * len(boxes))  # forzamos la etiqueta pedida
    return all_boxes, all_labels

TEST_IMAGES = Path("/content/Construction-Site-Safety-27/test/images")
TEST_LABELS = Path("/content/Construction-Site-Safety-27/test/labels")

# IDs de clase según data.yaml (mismos usados en las secciones anteriores)
HARDHAT_CLASS_ID = 0
NO_HARDHAT_CLASS_ID = 2
PERSON_CLASS_ID = 5

# Warm-up (no se mide)
first_image_path = sorted(TEST_IMAGES.glob("*"))[0]
_ = run_florence_multi(Image.open(first_image_path).convert("RGB"), ["hard hat", "bare head", "person"])
print("Calentamiento (WARM-UP) de Florence-2 completado.")

Calentamiento (WARM-UP) de Florence-2 completado.


Medición (inferencia + tiempo, en un solo paso)

In [ ]:
import time
import numpy as np

florence_predictions = []  # cache de (path, boxes, labels) para reutilizar en mAP y evento
times_florence = []

for img_path in sorted(TEST_IMAGES.glob("*")):
    image = Image.open(img_path).convert("RGB")

    start = time.perf_counter()
    boxes, labels = run_florence_multi(image, ["hard hat", "bare head", "person"])
    end = time.perf_counter()

    times_florence.append(end - start)
    florence_predictions.append({
        "path": img_path,
        "width": image.width,
        "height": image.height,
        "boxes": boxes,
        "labels": [l.lower().strip() for l in labels],
    })

print(f"Inferencia completa: {len(florence_predictions)} frames procesados con Florence-2.")

Inferencia completa: 82 frames procesados con Florence-2.


## métricas (F1 a nivel-evento)

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

y_true_florence = []
y_pred_florence = []

for pred in florence_predictions:
    label_path = TEST_LABELS / (pred["path"].stem + ".txt")

    # --- Ground truth ---
    gt_no_hardhat = False
    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                if int(line.split()[0]) == NO_HARDHAT_CLASS_ID:
                    gt_no_hardhat = True
                    break

    # --- Normalización de clases / lógica de negación (misma que YOLO-World) ---
    # Marcamos el frame como positivo para la infracción si Florence-2 detecta "head"
    # (cabeza sin casco) o "person" sin "hard hat" en la misma escena.
    pred_labels = pred["labels"]
    pred_no_hardhat = False
    if "bare head" in pred_labels or ("person" in pred_labels and "hard hat" not in pred_labels):
        pred_no_hardhat = True

    y_true_florence.append(gt_no_hardhat)
    y_pred_florence.append(pred_no_hardhat)

tn_f, fp_f, fn_f, tp_f = confusion_matrix(y_true_florence, y_pred_florence).ravel()
precision_f = precision_score(y_true_florence, y_pred_florence, zero_division=0)
recall_f = recall_score(y_true_florence, y_pred_florence, zero_division=0)
f1_f = f1_score(y_true_florence, y_pred_florence, zero_division=0)

print("--- MÉTRICAS (METRICS) A NIVEL-EVENTO (FLORENCE-2) ---")
print(f"TP : {tp_f} | FP : {fp_f} | TN : {tn_f} | FN : {fn_f}")
print(f"Precisión (PRECISION): {precision_f:.4f}")
print(f"Sensibilidad (RECALL): {recall_f:.4f}")
print(f"Puntuación F1 (F1 SCORE): {f1_f:.4f}")

--- MÉTRICAS (METRICS) A NIVEL-EVENTO (FLORENCE-2) ---
TP : 25 | FP : 57 | TN : 0 | FN : 0
Precisión (PRECISION): 0.3049
Sensibilidad (RECALL): 1.0000
Puntuación F1 (F1 SCORE): 0.4673


In [ ]:
img_debug = Image.open(sorted(TEST_IMAGES.glob("*"))[0]).convert("RGB")

boxes_head, labels_head = run_florence_detection(img_debug, "head")
boxes_bare, labels_bare = run_florence_detection(img_debug, "bare head")

print("head:", boxes_head, labels_head)
print("bare head:", boxes_bare, labels_bare)

head: [[322.8800048828125, 365.1199951171875, 351.67999267578125, 393.2799987792969]] ['head']
bare head: [[322.239990234375, 365.1199951171875, 351.67999267578125, 393.91998291015625]] ['bare head']


Florence-2 con <OPEN_VOCABULARY_DETECTION> detecta la misma región para "head" y "bare head" (cajas idénticas, ver debug), evidenciando que el modelo ancla sustantivos pero no procesa atributos negativos. Esto genera un F1 de evento de 0.4673 con Recall=1.0 pero Precisión=0.30 — el sistema declara "sin casco" en cada frame porque cualquier cabeza visible (con o sin casco) dispara la alerta. A diferencia de YOLO-World, que al menos combina la ausencia de "hard hat" con presencia de "person" para inferir la negación indirectamente, Florence-2 no tiene ese mecanismo de comparación cuando se le pide una sola entidad.

## Evaluación de cajas (mAP@0.5 aproximado) en clases compartibles


In [ ]:
import numpy as np

def yolo_txt_to_xyxy(label_path, target_class_id, img_w, img_h):
    """Lee un label .txt en formato YOLO y devuelve cajas xyxy en píxeles para una clase."""
    boxes = []
    if not label_path.exists():
        return boxes
    with open(label_path, "r") as f:
        for line in f:
            parts = line.split()
            cls_id = int(parts[0])
            if cls_id != target_class_id:
                continue
            xc, yc, w, h = map(float, parts[1:5])
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            boxes.append([x1, y1, x2, y2])
    return boxes

def iou_xyxy(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b
    inter_x1, inter_y1 = max(xa1, xb1), max(ya1, yb1)
    inter_x2, inter_y2 = min(xa2, xb2), min(ya2, yb2)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area_a = max(0, xa2 - xa1) * max(0, ya2 - ya1)
    area_b = max(0, xb2 - xb1) * max(0, yb2 - yb1)
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0

def evaluar_clase_iou(class_label_florence, class_id_gt, iou_thr=0.5):
    tp, fp, fn = 0, 0, 0
    for pred in florence_predictions:
        label_path = TEST_LABELS / (pred["path"].stem + ".txt")
        gt_boxes = yolo_txt_to_xyxy(label_path, class_id_gt, pred["width"], pred["height"])

        pred_boxes = [b for b, l in zip(pred["boxes"], pred["labels"]) if l == class_label_florence]

        matched_gt = set()
        for pb in pred_boxes:
            best_iou, best_idx = 0.0, -1
            for i, gb in enumerate(gt_boxes):
                if i in matched_gt:
                    continue
                iou = iou_xyxy(pb, gb)
                if iou > best_iou:
                    best_iou, best_idx = iou, i
            if best_iou >= iou_thr:
                tp += 1
                matched_gt.add(best_idx)
            else:
                fp += 1
        fn += len(gt_boxes) - len(matched_gt)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

print("--- PROXY AP@0.5 CLASES COMPARTIBLES (FLORENCE-2) ---")
print(f"{'Clase (CLASS)':20} {'Precisión':>10} {'Recall':>10} {'F1 (proxy AP)':>15}")

for label_texto, class_id, nombre in [("hard hat", HARDHAT_CLASS_ID, "Hardhat"),
                                        ("person", PERSON_CLASS_ID, "Person")]:
    p, r, f1 = evaluar_clase_iou(label_texto, class_id)
    print(f"{nombre:20} {p:10.4f} {r:10.4f} {f1:15.4f}")

--- PROXY AP@0.5 CLASES COMPARTIBLES (FLORENCE-2) ---
Clase (CLASS)         Precisión     Recall   F1 (proxy AP)
Hardhat                  0.4286     0.4364          0.4324
Person                   0.7469     0.6954          0.7202


## Velocidad

In [ ]:
# Reutilizamos los tiempos ya medidos durante la inferencia principal
# (no volvemos a correr el modelo: sería doble costo en un VLM lento en T4).
latency_florence = np.mean(times_florence)
fps_florence = 1 / latency_florence

print("="*40)
print("VELOCIDAD DEL MODELO FLORENCE-2")
print("="*40)
print(f"Frames evaluados : {len(times_florence)}")
print(f"Latencia promedio: {latency_florence:.4f} s/frame")
print(f"FPS promedio     : {fps_florence:.2f}")

VELOCIDAD DEL MODELO FLORENCE-2
Frames evaluados : 82
Latencia promedio: 0.5522 s/frame
FPS promedio     : 1.81


## Robustez

In [ ]:
# Sensibilidad al prompt (misma idea que en YOLO-World): variamos la redacción
# del texto de entrada 3-5 veces y medimos cómo cambia el F1 a nivel-evento.
# Cada variante es una LISTA de términos (no un string con puntos), porque
# OPEN_VOCABULARY_DETECTION trata un string completo como una sola frase a
# buscar, no como varias clases — por eso usamos run_florence_multi, que
# llama al modelo una vez por término y fusiona los resultados.
#
# Usamos una MUESTRA del test set (no el set completo) porque Florence-2 es
# demasiado lento para correr varias pasadas completas en Colab dentro del
# tiempo disponible; lo documentamos explícitamente en el informe.

prompts_alternativos_florence = [
    ["hard hat", "head", "person"],
    ["safety helmet", "bare head", "human"],
    ["cap", "person"],
    ["protective headwear", "unprotected head"],
]

N_MUESTRA = 60  # tamaño de la muestra para la prueba de robustez
muestra_paths = sorted(TEST_IMAGES.glob("*"))[:N_MUESTRA]

# Ground truth de la muestra (igual lógica que antes)
y_true_muestra = []
for img_path in muestra_paths:
    label_path = TEST_LABELS / (img_path.stem + ".txt")
    gt_no_hardhat = False
    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                if int(line.split()[0]) == NO_HARDHAT_CLASS_ID:
                    gt_no_hardhat = True
                    break
    y_true_muestra.append(gt_no_hardhat)

resultados_sensibilidad_florence = {}

for prompt_lista in prompts_alternativos_florence:
    y_pred_variant = []
    casco_syn, cabeza_syn = prompt_lista[0], prompt_lista[1]
    persona_syn = prompt_lista[2] if len(prompt_lista) > 2 else "person"

    for img_path in muestra_paths:
        image = Image.open(img_path).convert("RGB")
        _, labels = run_florence_multi(image, prompt_lista)
        labels = [l.lower().strip() for l in labels]

        pred_no_hardhat = cabeza_syn in labels or (persona_syn in labels and casco_syn not in labels)
        y_pred_variant.append(pred_no_hardhat)

    print(f"Variante {prompt_lista} -> primeras 10 predicciones: {y_pred_variant[:10]}")   # <-- agregado
    f1_variante = f1_score(y_true_muestra, y_pred_variant, zero_division=0)
    ...
    resultados_sensibilidad_florence[str(prompt_lista)] = round(f1_variante, 4)
    print(f"✅ Variante evaluada {prompt_lista} -> F1 Score: {f1_variante:.4f}")

print("\n--- RESUMEN DE SENSIBILIDAD AL PROMPT (FLORENCE-2, muestra de", N_MUESTRA, "frames) ---")
print(resultados_sensibilidad_florence)

Variante ['hard hat', 'head', 'person'] -> primeras 10 predicciones: [True, True, True, True, True, True, True, True, True, True]
✅ Variante evaluada ['hard hat', 'head', 'person'] -> F1 Score: 0.5185
Variante ['safety helmet', 'bare head', 'human'] -> primeras 10 predicciones: [True, True, True, True, True, True, True, True, True, True]
✅ Variante evaluada ['safety helmet', 'bare head', 'human'] -> F1 Score: 0.5185
Variante ['cap', 'person'] -> primeras 10 predicciones: [True, True, True, True, True, True, True, True, True, True]
✅ Variante evaluada ['cap', 'person'] -> F1 Score: 0.5185
Variante ['protective headwear', 'unprotected head'] -> primeras 10 predicciones: [True, True, True, True, True, True, True, True, True, True]
✅ Variante evaluada ['protective headwear', 'unprotected head'] -> F1 Score: 0.5185

--- RESUMEN DE SENSIBILIDAD AL PROMPT (FLORENCE-2, muestra de 60 frames) ---
{"['hard hat', 'head', 'person']": 0.5185, "['safety helmet', 'bare head', 'human']": 0.5185, "['cap

# Moondream2 (VLM generativo)

## Instalación

In [ ]:
# Moondream2 (vikhyatk, licencia Apache-2.0). Usamos la misma versión de
# transformers ya fijada para Florence-2 (4.49.0) para no generar otro
# conflicto de dependencias en la misma sesión de Colab.
!pip install -q einops pyvips-binary
!pip install -q "transformers==4.49.0" "huggingface_hub==0.34.0" accelerate

# Si esta celda instala algo nuevo (pyvips-binary), reinicia sesión igual
# que hicimos con Florence-2 antes de seguir.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 64.8 MB/s eta 0:00:00


## Cargamos modelo

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Fijamos una revisión concreta del modelo (no "main") para que el notebook
# sea reproducible: si el repo de HF cambia el código remoto más adelante,
# esta celda sigue funcionando igual.
MOONDREAM_MODEL_ID = "vikhyatk/moondream2"
MOONDREAM_REVISION = "2024-08-26"

model_moondream = AutoModelForCausalLM.from_pretrained(
    MOONDREAM_MODEL_ID,
    revision=MOONDREAM_REVISION,
    trust_remote_code=True,
    torch_dtype=torch_dtype,
).to(device).eval()

tokenizer_moondream = AutoTokenizer.from_pretrained(MOONDREAM_MODEL_ID, revision=MOONDREAM_REVISION)

print(f"Modelo Moondream2 ({MOONDREAM_REVISION}) cargado en {device}. Licencia: Apache-2.0.")

config.json:   0%|          | 0.00/319 [00:00<?, ?B/s]

configuration_moondream.py: 0.00B [00:00, ?B/s]

moondream.py: 0.00B [00:00, ?B/s]

region_model.py: 0.00B [00:00, ?B/s]

fourier_features.py:   0%|          | 0.00/558 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Modelo Moondream2 (2024-08-26) cargado en cuda. Licencia: Apache-2.0.


## Definición de pregunta en lenguaje natural

In [ ]:
# Pregunta binaria por frame, tal como la plantea la guía.
pregunta_moondream = "Is everyone in the image wearing a hard hat / helmet?"

def preguntar_moondream(image, pregunta, model=model_moondream, tokenizer=tokenizer_moondream):
    """Codifica la imagen y le hace la pregunta al modelo. Devuelve la respuesta en texto."""
    enc_image = model.encode_image(image)
    respuesta = model.answer_question(enc_image, pregunta, tokenizer)
    return respuesta

def respuesta_a_booleano(respuesta_texto):
    """Convierte la respuesta libre del modelo en 'sin casco' (True) / 'con casco' (False).

    Moondream2 responde en lenguaje natural (ej. 'No, one person is not wearing a
    helmet.'), no un booleano limpio. Buscamos la negación en el texto: si la
    respuesta empieza en 'no' o contiene alguna señal de negación, interpretamos
    que SÍ hay alguien sin casco (evento positivo de infracción)."""
    texto = respuesta_texto.strip().lower()
    negaciones = ["no,", "no.", "not everyone", "not all", "isn't wearing", "is not wearing", "without a"]
    if texto.startswith("no") or any(neg in texto for neg in negaciones):
        return True  # hay alguien sin casco
    return False  # todos llevan casco (según el modelo)

print("Pregunta configurada:", pregunta_moondream)

Pregunta configurada: Is everyone in the image wearing a hard hat / helmet?


## Inferencia sobre el conjunto de test

warm-up

In [ ]:
first_image_moondream = Image.open(sorted(TEST_IMAGES.glob("*"))[0]).convert("RGB")
_ = preguntar_moondream(first_image_moondream, pregunta_moondream)
print("Calentamiento (WARM-UP) de Moondream2 completado.")

Calentamiento (WARM-UP) de Moondream2 completado.


Medición (inferencia + tiempo, en un solo paso, igual que en Florence-2)

In [ ]:
moondream_predictions = []  # cache de (path, respuesta cruda, booleano parseado)
times_moondream = []

for img_path in sorted(TEST_IMAGES.glob("*")):
    image = Image.open(img_path).convert("RGB")

    start = time.perf_counter()
    respuesta = preguntar_moondream(image, pregunta_moondream)
    end = time.perf_counter()

    times_moondream.append(end - start)
    moondream_predictions.append({
        "path": img_path,
        "respuesta_cruda": respuesta,
        "pred_no_hardhat": respuesta_a_booleano(respuesta),
    })

print(f"Inferencia completa: {len(moondream_predictions)} frames procesados con Moondream2.")

# Vistazo rápido a respuestas crudas, para verificar que el parseo tiene sentido
for p in moondream_predictions[:5]:
    print(f"  {p['path'].name}: \"{p['respuesta_cruda']}\" -> {p['pred_no_hardhat']}")

Inferencia completa: 82 frames procesados con Moondream2.
  -4405-_png_jpg.rf.82b5c10b2acd1cfaa24259ada8e599fe.jpg: "Yes, everyone in the image is wearing a hard hat and a helmet, indicating that they are working in a construction or industrial environment." -> False
  000005_jpg.rf.96e9379ccae638140c4a90fc4b700a2b.jpg: "Yes, both men in the image are wearing hard hats and helmets while working on the electrical equipment." -> False
  002551_jpg.rf.ce4b9f934161faa72c80dc6898d37b2d.jpg: "Yes, everyone in the image is wearing a hard hat and a helmet, indicating that they are construction workers." -> False
  003357_jpg.rf.9867f91e88089bb68dc95947d5116d14.jpg: "Yes, everyone in the image is wearing a hard hat and a helmet." -> False
  004063_jpg.rf.1b7cdc4035bcb24ef69b8798b444053e.jpg: "Yes, everyone in the image is wearing hard hats or helmets, indicating that they are workers or involved in some kind of construction or maintenance activity." -> False


## métricas (F1 a nivel-evento)

In [ ]:
y_true_moondream = []
y_pred_moondream = []

for pred in moondream_predictions:
    label_path = TEST_LABELS / (pred["path"].stem + ".txt")

    gt_no_hardhat = False
    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                if int(line.split()[0]) == NO_HARDHAT_CLASS_ID:
                    gt_no_hardhat = True
                    break

    y_true_moondream.append(gt_no_hardhat)
    y_pred_moondream.append(pred["pred_no_hardhat"])

tn_m, fp_m, fn_m, tp_m = confusion_matrix(y_true_moondream, y_pred_moondream).ravel()
precision_m = precision_score(y_true_moondream, y_pred_moondream, zero_division=0)
recall_m = recall_score(y_true_moondream, y_pred_moondream, zero_division=0)
f1_m = f1_score(y_true_moondream, y_pred_moondream, zero_division=0)

print("--- MÉTRICAS (METRICS) A NIVEL-EVENTO (MOONDREAM2) ---")
print(f"TP : {tp_m} | FP : {fp_m} | TN : {tn_m} | FN : {fn_m}")
print(f"Precisión (PRECISION): {precision_m:.4f}")
print(f"Sensibilidad (RECALL): {recall_m:.4f}")
print(f"Puntuación F1 (F1 SCORE): {f1_m:.4f}")

--- MÉTRICAS (METRICS) A NIVEL-EVENTO (MOONDREAM2) ---
TP : 20 | FP : 31 | TN : 26 | FN : 5
Precisión (PRECISION): 0.3922
Sensibilidad (RECALL): 0.8000
Puntuación F1 (F1 SCORE): 0.5263


## Velocidad

In [ ]:
latency_moondream = np.mean(times_moondream)
fps_moondream = 1 / latency_moondream

print("="*40)
print("VELOCIDAD DEL MODELO MOONDREAM2")
print("="*40)
print(f"Frames evaluados : {len(times_moondream)}")
print(f"Latencia promedio: {latency_moondream:.4f} s/frame")
print(f"FPS promedio     : {fps_moondream:.2f}")

VELOCIDAD DEL MODELO MOONDREAM2
Frames evaluados : 82
Latencia promedio: 1.1810 s/frame
FPS promedio     : 0.85


## Robustez

In [ ]:
# Sensibilidad a la redacción de la pregunta (3-5 variantes), igual que en
# Florence-2 y YOLO-World. Usamos la misma muestra de N_MUESTRA frames para
# poder comparar directamente contra los resultados de Florence-2.

preguntas_alternativas_moondream = [
    "Is everyone in the image wearing a hard hat / helmet?",
    "Are all the workers wearing safety helmets?",
    "Is there any person without a hard hat in this picture?",
    "Does everyone have proper head protection on?",
]

resultados_sensibilidad_moondream = {}

for pregunta in preguntas_alternativas_moondream:
    y_pred_variant = []
    for img_path in muestra_paths:
        image = Image.open(img_path).convert("RGB")
        respuesta = preguntar_moondream(image, pregunta)
        y_pred_variant.append(respuesta_a_booleano(respuesta))

    f1_variante = f1_score(y_true_muestra, y_pred_variant, zero_division=0)
    resultados_sensibilidad_moondream[pregunta] = round(f1_variante, 4)
    print(f"✅ Pregunta evaluada \"{pregunta}\" -> F1 Score: {f1_variante:.4f}")

print("\n--- RESUMEN DE SENSIBILIDAD A LA PREGUNTA (MOONDREAM2, muestra de", N_MUESTRA, "frames) ---")
print(resultados_sensibilidad_moondream)

✅ Pregunta evaluada "Is everyone in the image wearing a hard hat / helmet?" -> F1 Score: 0.6129
✅ Pregunta evaluada "Are all the workers wearing safety helmets?" -> F1 Score: 0.5714
✅ Pregunta evaluada "Is there any person without a hard hat in this picture?" -> F1 Score: 0.5250
✅ Pregunta evaluada "Does everyone have proper head protection on?" -> F1 Score: 0.5588

--- RESUMEN DE SENSIBILIDAD A LA PREGUNTA (MOONDREAM2, muestra de 60 frames) ---
{'Is everyone in the image wearing a hard hat / helmet?': 0.6129, 'Are all the workers wearing safety helmets?': 0.5714, 'Is there any person without a hard hat in this picture?': 0.525, 'Does everyone have proper head protection on?': 0.5588}


## Prueba

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path

# ===========================
# CONFIGURACIÓN
# ===========================

N_FRAMES = 20

image_paths = sorted(TEST_IMAGES.glob("*"))[:N_FRAMES]

# ===========================
# FUNCIÓN PARA DIBUJAR BOXES
# ===========================

def draw_boxes_florence(image, boxes, labels):

    img = image.copy()
    draw = ImageDraw.Draw(img)

    for box, label in zip(boxes, labels):

        x1, y1, x2, y2 = map(int, box)

        draw.rectangle(
            [x1, y1, x2, y2],
            outline="red",
            width=3
        )

        draw.text(
            (x1, max(0, y1-15)),
            label,
            fill="red"
        )

    return np.array(img)


# ===========================
# DEMOSTRACIÓN
# ===========================

for i, img_path in enumerate(image_paths):

    print(f"\n================ FRAME {i+1}/{len(image_paths)} ================\n")

    pil_img = Image.open(img_path).convert("RGB")

    # -------------------------------------------------
    # YOLOv8
    # -------------------------------------------------

    yolo_result = model.predict(
        source=str(img_path),
        verbose=False
    )[0]

    img_yolo = cv2.cvtColor(
        yolo_result.plot(),
        cv2.COLOR_BGR2RGB
    )

    # -------------------------------------------------
    # YOLO WORLD
    # -------------------------------------------------

    world_result = model_world.predict(
        source=str(img_path),
        verbose=False
    )[0]

    img_world = cv2.cvtColor(
        world_result.plot(),
        cv2.COLOR_BGR2RGB
    )

    # -------------------------------------------------
    # FLORENCE
    # -------------------------------------------------

    boxes, labels = run_florence_detection(
        pil_img,
        text_input_florence
    )

    img_florence = draw_boxes_florence(
        pil_img,
        boxes,
        labels
    )

    # -------------------------------------------------
    # MOONDREAM
    # -------------------------------------------------

    respuesta = preguntar_moondream(
        pil_img,
        pregunta_moondream
    )

    img_moondream = np.array(pil_img)

    # -------------------------------------------------
    # MOSTRAR
    # -------------------------------------------------

    fig, ax = plt.subplots(
        2,
        2,
        figsize=(14,10)
    )

    fig.suptitle(
        f"FRAME {i+1}",
        fontsize=18,
        fontweight="bold"
    )

    ax[0,0].imshow(img_yolo)
    ax[0,0].set_title("YOLOv8")
    ax[0,0].axis("off")

    ax[0,1].imshow(img_world)
    ax[0,1].set_title("YOLO-World")
    ax[0,1].axis("off")

    ax[1,0].imshow(img_florence)
    ax[1,0].set_title("Florence-2")
    ax[1,0].axis("off")

    ax[1,1].imshow(img_moondream)
    ax[1,1].set_title("Moondream2")
    ax[1,1].axis("off")

    ax[1,1].text(
        0.02,
        0.02,
        respuesta,
        transform=ax[1,1].transAxes,
        fontsize=10,
        color="yellow",
        bbox=dict(facecolor="black", alpha=0.75)
    )

    plt.tight_layout()
    plt.show()

NameError: name 'TEST_IMAGES' is not defined